## 1. Imports

In [1]:
# Cell 1: Imports
import numpy as np
import pandas as pd
import re
import matplotlib.pyplot as plt
import os
import joblib


from collections import defaultdict
from scipy.stats import zscore
# Scikit-learn & Survival Analysis
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, KFold
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer, SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.ensemble import RandomSurvivalForest
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
from sklearn.ensemble import RandomForestRegressor
from sksurv.metrics import concordance_index_ipcw
from sksurv.util import Surv



# Load Data
df = pd.read_csv("./data/X_train/clinical_train.csv")
df_eval = pd.read_csv("./data/X_test/clinical_test.csv")

maf_df = pd.read_csv("./data/X_train/molecular_train.csv")
maf_eval = pd.read_csv("./data/X_test/molecular_test.csv")

target_df = pd.read_csv("./data/target_train.csv")

print("Data Loaded.")

Data Loaded.


## 2. Target Cleaning

In [2]:
# Cell 2: Target Cleaning
target_df.dropna(subset=['OS_YEARS', 'OS_STATUS'], inplace=True)
target_df['OS_YEARS'] = pd.to_numeric(target_df['OS_YEARS'], errors='coerce')
target_df['OS_STATUS'] = target_df['OS_STATUS'].astype(bool)

# Merge target into training data
df = df.merge(target_df[['ID', 'OS_YEARS', 'OS_STATUS']], on='ID', how='inner')

## 3. Parsing functions

In [3]:
# Cell 3: Parsing Functions

def parse_PROTEIN_CHANGE(protein):
    """Extracts the numeric position (hotspot) from protein change string."""
    protein = str(protein)
    if len(protein) == 0 or protein == 'nan':
        return 0
    # Regex to find the number in p.R882H -> 882
    match = re.search(r"(\d+)", protein)
    if match:
        return int(match.group(1))
    return 0

def parse_GENE(gene):
    """Returns the full gene name in uppercase."""
    gene = str(gene)
    if len(gene) == 0 or gene == 'nan':
        return "UNKNOWN"
    return gene.upper()

def parse_CYTO(iscn):
    """Parses ISCN strings for complex karyotypes and translocations."""
    iscn = str(iscn).upper().replace(" ", "")
    results = defaultdict(int)
    
    # 1. Detect Complex Karyotype
    clones = iscn.split("/")
    max_abnormalities = 0
    for clone in clones:
        abnormalities = re.findall(r"([+-]\d+|DEL|ADD|INV|T\(|DER)", clone)
        if len(abnormalities) > max_abnormalities:
            max_abnormalities = len(abnormalities)
    if max_abnormalities >= 3:
        results["Complex_Karyotype"] = 1

    # 2. Specific Translocations/Inversions
    structural = re.findall(r"(T|INV)\((\d+|X|Y)[;]?(\d+|X|Y)?\)", iscn)
    for type_, chrom1, chrom2 in structural:
        chrom2_str = f";{chrom2}" if chrom2 else ""
        key = f"{type_}({chrom1}{chrom2_str})"
        results[key] = 1

    # 3. Numeric changes
    numeric_changes = re.findall(r"(?<![0-9])([+-])(\d+|X|Y)(?=[,/]|$)", iscn)
    for sign, num in numeric_changes:
        key = f"{sign}{num}"
        results[key] = 1

    if not results:
        results["normal"] = 1
    return dict(results)

## 4. Data processing

In [4]:
# Cell 4: Cytogenetics Processing (Fixed Fragmentation)

# Calculate Nmut
if 'Nmut' not in df.columns:
    nmut_train = maf_df.groupby('ID').size().reset_index(name='Nmut')
    nmut_test = maf_eval.groupby('ID').size().reset_index(name='Nmut')
    df = df.merge(nmut_train, on='ID', how='left').fillna({'Nmut': 0})
    df_eval = df_eval.merge(nmut_test, on='ID', how='left').fillna({'Nmut': 0})

total_features = ['Nmut']

# Parse CYTO - FIXED TO USE CONCAT INSTEAD OF FRAGMENTED ASSIGNMENT
for i, dataset in enumerate([df, df_eval]):
    cyto_dicts = dataset['CYTOGENETICS'].apply(parse_CYTO)
    cyto_df = pd.DataFrame(cyto_dicts.tolist(), index=dataset.index).fillna(0)
    
    # Use concat to avoid fragmentation
    if i == 0:
        df = pd.concat([df, cyto_df], axis=1)
    else:
        df_eval = pd.concat([df_eval, cyto_df], axis=1)

# Add only columns relevant to training data
cyto_cols = [c for c in df.columns if any(x in c for x in ['+', '-', 't(', 'inv(', 'Complex', 'normal'])]
total_features.extend(cyto_cols)
total_features.remove('-72')


## 5. Molecular features with VAF weighting

In [5]:
# Cell 5: Molecular Features (Optimized)

def process_molecular_features():
    global df, df_eval, total_features
    
    # 1. GENE features (Weighted by Max VAF)
    print("Processing Genes...")
    maf_df['GENE_CLEAN'] = maf_df['GENE'].apply(parse_GENE)
    maf_eval['GENE_CLEAN'] = maf_eval['GENE'].apply(parse_GENE)
    
    gene_pivot_train = maf_df.pivot_table(index='ID', columns='GENE_CLEAN', values='VAF', aggfunc='max', fill_value=0)
    gene_pivot_test = maf_eval.pivot_table(index='ID', columns='GENE_CLEAN', values='VAF', aggfunc='max', fill_value=0)
    
    # Rename columns
    gene_pivot_train.columns = [f"GENE_{c}" for c in gene_pivot_train.columns]
    gene_pivot_test.columns = [f"GENE_{c}" for c in gene_pivot_test.columns]
    
    # Align columns: Ensure test has exact same columns as train
    gene_pivot_test = gene_pivot_test.reindex(columns=gene_pivot_train.columns, fill_value=0)
            
    # Merge
    df = df.merge(gene_pivot_train, on='ID', how='left').fillna(0)
    df_eval = df_eval.merge(gene_pivot_test, on='ID', how='left').fillna(0)
    
    total_features.extend(gene_pivot_train.columns.tolist())
    
    # 2. EFFECT and PROTEIN_CHANGE (One-Hot)
    for feature in ['EFFECT', 'PROTEIN_CHANGE']:
        print(f"Processing {feature}...")
        
        if feature == 'PROTEIN_CHANGE':
            maf_df['temp_feat'] = maf_df[feature].apply(parse_PROTEIN_CHANGE)
            maf_eval['temp_feat'] = maf_eval[feature].apply(parse_PROTEIN_CHANGE)
        else:
            maf_df['temp_feat'] = maf_df[feature]
            maf_eval['temp_feat'] = maf_eval[feature]
            
        dummies_train = pd.get_dummies(maf_df['temp_feat'], prefix=feature)
        dummies_train['ID'] = maf_df['ID']
        agg_train = dummies_train.groupby('ID').max()
        
        dummies_test = pd.get_dummies(maf_eval['temp_feat'], prefix=feature)
        dummies_test['ID'] = maf_eval['ID']
        agg_test = dummies_test.groupby('ID').max()
        
        # Align columns
        agg_test = agg_test.reindex(columns=agg_train.columns, fill_value=0)
        
        # Merge
        df = df.merge(agg_train, on='ID', how='left').fillna(0)
        df_eval = df_eval.merge(agg_test, on='ID', how='left').fillna(0)
        
        total_features.extend(list(agg_train.columns))

process_molecular_features()

Processing Genes...
Processing EFFECT...
Processing PROTEIN_CHANGE...


## 6. Interactions and clinical preparation

In [6]:
# Cell 6: Interactions & Clinical Data (Updated with IterativeImputer)

def add_interactions(data):
    if 'GENE_NPM1' in data.columns and 'GENE_FLT3' in data.columns:
        data['INT_NPM1_pos_FLT3_neg'] = ((data['GENE_NPM1'] > 0) & (data['GENE_FLT3'] == 0)).astype(int)
    if 'GENE_TP53' in data.columns and 'Complex_Karyotype' in data.columns:
        data['INT_TP53_Complex'] = ((data['GENE_TP53'] > 0) & (data['Complex_Karyotype'] > 0)).astype(int)
    return data

df = add_interactions(df)
df_eval = add_interactions(df_eval)

# Add interactions if they exist
for feat in ['INT_NPM1_pos_FLT3_neg', 'INT_TP53_Complex']:
    if feat in df.columns:
        total_features.append(feat)

# Clinical Transforms
clinical_cols = ['BM_BLAST', 'WBC', 'ANC', 'PLT', 'HB', 'MONOCYTES']
for col in clinical_cols:
    # Log transform first to normalize skewed distributions before imputation
    df[col] = np.log1p(df[col])
    df_eval[col] = np.log1p(df_eval[col])

# --- CHANGED: Iterative Imputation ---
print("Starting Iterative Imputation on clinical variables...")

# We use a lightweight Random Forest estimator inside MICE. 
# This learns non-linear relationships (e.g., WBC vs Blasts) better than BayesianRidge.
knn_imputer = KNNImputer(n_neighbors=5, weights='uniform')  

# Apply ONLY to clinical columns to avoid crashing on the 1000+ gene columns
df[clinical_cols] = knn_imputer.fit_transform(df[clinical_cols])
df_eval[clinical_cols] = knn_imputer.transform(df_eval[clinical_cols])
print("Imputation Complete.")

total_features.extend(clinical_cols)
# Remove duplicates
total_features = list(set([f for f in total_features if f in df.columns]))
print(f"Total Features: {len(total_features)}")

scaler = StandardScaler()
df[total_features] = scaler.fit_transform(df[total_features])
df_eval[total_features] = scaler.transform(df_eval[total_features])

# Prepare Data
X = df[total_features]
y = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', df)

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Data split complete.")

Starting Iterative Imputation on clinical variables...
Imputation Complete.
Total Features: 1772
Data split complete.


## 8. Random Survival Forest

In [30]:
# Cell 8: Random Survival Forest
from re import split
from sklearn.model_selection import StratifiedKFold

N_fit = 5
train_scores = []
test_scores = []
print("-" * 40)
print(f"{'Fold':<5} | {'Train':<10} | {'Test':<10}")
print("-" * 40)

# Use event status for stratification (convert bool -> int)
y_event = y['OS_STATUS'].astype(int)
skf = StratifiedKFold(n_splits=N_fit, shuffle=True, random_state=42)

rsf = RandomSurvivalForest(
        n_estimators=1000,
        min_samples_leaf=9,
        max_features='sqrt',
        max_depth=25,
        random_state=42,
        verbose=0
    )

for loop, (train_idx, test_idx) in enumerate(skf.split(X.values, y_event)):
    # Proper DataFrame indexing with iloc
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # RSF doesn't need scaling, but it's fine if we pass raw X_train
    rsf.fit(X_train, y_train)

    c_index_rsf_train = concordance_index_ipcw(y_train, y_train, rsf.predict(X_train), tau=1)[0]
    c_index_rsf = concordance_index_ipcw(y_train, y_test, rsf.predict(X_test), tau=1)[0]
    train_scores.append(c_index_rsf_train)
    test_scores.append(c_index_rsf)
    print(f"{loop+1:<5} | {c_index_rsf_train:.4f}     | {c_index_rsf:.4f}")
print(f"Train score: {np.mean(train_scores)} +/- {np.std(train_scores)}")
print(f"Test score: {np.mean(test_scores)} +/- {np.std(test_scores)}")


----------------------------------------
Fold  | Train      | Test      
----------------------------------------


KeyboardInterrupt: 

In [44]:
# Cell 8: Random Survival Forest
N_fit = 5
train_scores = []
test_scores = []
print("-" * 40)
print(f"{'Fold':<5} | {'C-index train':<10} | {'C-index test':<10}")
print("-" * 40)

skf = StratifiedKFold(n_splits=N_fit, shuffle=True, random_state=42)
for loop, (train_idx, test_idx) in enumerate(skf.split(X.values, y_event)):
    # Proper DataFrame indexing with iloc
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # RSF doesn't need scaling, but it's fine if we pass raw X_train
    gbsa = GradientBoostingSurvivalAnalysis(
        loss='coxph',            # Loss function
        n_estimators=400,        # Number of trees
        learning_rate=0.01,      # Learning rate
        max_depth=50,             # Maximum depth of trees
        min_samples_split=3,    # Minimum samples required to split an internal node
        min_samples_leaf=3,     # Minimum samples required in a leaf node
        max_features='sqrt',     # Number of features to consider at each split
        subsample=0.9,           # Fraction of samples used for training each tree
        random_state=42,       # Random seed for reproducibility
    )
    gbsa.fit(X_train, y_train)

    c_index_rsf_train = concordance_index_ipcw(y_train, y_train, gbsa.predict(X_train), tau=1)[0]
    c_index_rsf = concordance_index_ipcw(y_train, y_test, gbsa.predict(X_test), tau=1)[0]
    train_scores.append(c_index_rsf_train)
    test_scores.append(c_index_rsf)
    print(f"{loop+1:<5} | {c_index_rsf_train:.4f}     | {c_index_rsf:.4f}")
print(f"Train score: {np.mean(train_scores)} +/- {np.std(train_scores)}")
print(f"Test score: {np.mean(test_scores)} +/- {np.std(test_scores)}")

----------------------------------------
Fold  | C-index train | C-index test
----------------------------------------


KeyboardInterrupt: 

## 9. Submissions retrained on full dataset

### 9.1 RSF submission on full set

In [36]:
# Cell 11.1: Final "Production" Submission (Retraining on ALL data)

# 1. Prepare FULL Training Data (Combine train and local test)
X_full = df[total_features]
y_full = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', df)

# 2. Retrain the Best Model (RSF) on ALL Data
print("Retraining RSF on full dataset (100% of data)...")
rsf.fit(X_full, y_full)

# 3. Prepare Evaluation Data
# Align columns (fill missing with 0)
X_eval_final = df_eval.reindex(columns=total_features, fill_value=0)

# 4. Predict
final_predictions = rsf.predict(X_eval_final)

# 5. Save
submission = pd.DataFrame({
    'ID': df_eval['ID'],
    'risk_score': final_predictions
})

filename = "submission/submission_rsf_full_data.csv"
submission.to_csv(filename, index=False)
print(f"Saved {filename} - Ready for upload!")

Retraining RSF on full dataset (100% of data)...
Saved submission/submission_rsf_full_data.csv - Ready for upload!


### 9.2 GBSA submission on full set

In [45]:
# Cell 11.1: Final "Production" Submission (Retraining on ALL data)

# 1. Prepare FULL Training Data (Combine train and local test)
X_full = df[total_features]
y_full = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', df)

# 2. Retrain the Best Model (RSF) on ALL Data
print("Retraining GBSA on full dataset (100% of data)...")
gbsa.fit(X_full, y_full)

# 3. Prepare Evaluation Data
# Align columns (fill missing with 0)
X_eval_final = df_eval.reindex(columns=total_features, fill_value=0)

# 4. Predict
final_predictions = gbsa.predict(X_eval_final)

# 5. Save
submission = pd.DataFrame({
    'ID': df_eval['ID'],
    'risk_score': final_predictions
})

filename = "submission/submission_gbsa_full_data.csv"
submission.to_csv(filename, index=False)
print(f"Saved {filename} - Ready for upload!")

Retraining GBSA on full dataset (100% of data)...
Saved submission/submission_gbsa_full_data.csv - Ready for upload!


### 9.3 Ensemble model submission on full set

In [41]:
ensemble_scores = 0.6 * pd.read_csv("submission/submission_rsf_full_data.csv")['risk_score'] + 0.4 * pd.read_csv("submission/submission_gbsa_full_data.csv")['risk_score']

submission_ensemble = pd.DataFrame({
    'ID': df_eval['ID'],
    'risk_score': ensemble_scores
})

filename = "submission/submission_ensemble_full_data.csv"
submission_ensemble.to_csv(filename, index=False)
print(f"Saved {filename} - Ready for upload!")

Saved submission/submission_ensemble_full_data.csv - Ready for upload!
